In [1]:
import sys, warnings
sys.path.append("..")
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import statsmodels.api as sm
from pathlib import Path
from arch.bootstrap import MCS

df = pd.read_csv("../data/GSPC_processed.csv", index_col=0, parse_dates=True)
r = df["log_return"]

fc = pd.read_csv("../data/garch_forecasts.csv", index_col=0, parse_dates=True)
fc = fc.rename(columns=lambda c: c if c.startswith("Rolling 21d") else f"{c} (fixed)")

files = [("../data/rolling_garch_forecasts.csv", None),
         ("../data/lstm_forecasts.csv", "LSTM (old)"),
         ("../data/hybrid_forecasts.csv", "Hybrid (old)"),
         ("../data/lstm_qlike_forecasts.csv", "LSTM (QLIKE)"),
         ("../data/hybrid_qlike_forecasts.csv", "Hybrid (QLIKE)")]
for path, label in files:
    if Path(path).exists():
        x = pd.read_csv(path, index_col=0, parse_dates=True)
        if label:
            x = x.iloc[:, [0]]
            x.columns = [label]
        fc = fc.join(x, how="inner")

fc = fc.dropna().clip(lower=1e-8)
proxy = r.loc[fc.index] ** 2
print(fc.shape)
print(list(fc.columns))

(845, 11)
['GARCH(1,1) (fixed)', 'EGARCH (fixed)', 'GJR-GARCH (fixed)', 'Rolling 21d var', 'GARCH(1,1) (rolling)', 'EGARCH (rolling)', 'GJR-GARCH (rolling)', 'LSTM (old)', 'Hybrid (old)', 'LSTM (QLIKE)', 'Hybrid (QLIKE)']


In [2]:
# Per-day QLIKE loss for each model (lower is better)
losses = fc.apply(lambda f: np.log(f) + proxy / f)

mcs = MCS(losses, size=0.10, reps=2000, seed=42)
mcs.compute()
print("Models in the 90% Model Confidence Set:")
print(list(mcs.included))
print()
display(mcs.pvalues.round(3))

Models in the 90% Model Confidence Set:
['EGARCH (fixed)', 'EGARCH (rolling)', 'Hybrid (QLIKE)', 'LSTM (QLIKE)']



,Pvalue
Model name,
"GARCH(1,1) (fixed)",0.000
"GARCH(1,1) (rolling)",0.000
Hybrid (old),0.000
LSTM (old),0.000
Rolling 21d var,0.002
GJR-GARCH (fixed),0.036
GJR-GARCH (rolling),0.076
EGARCH (fixed),0.128
EGARCH (rolling),0.128


In [3]:
rows = {}
for name in fc.columns:
    X = sm.add_constant(fc[name].values)
    res = sm.OLS(proxy.values, X).fit(cov_type="HAC", cov_kwds={"maxlags": 5})
    wald = res.wald_test("const = 0, x1 = 1", scalar=True)
    rows[name] = {
        "intercept": res.params[0],
        "slope": res.params[1],
        "joint p (a=0, b=1)": float(wald.pvalue),
        "mean(r2 / forecast)": float((proxy / fc[name]).mean()),
    }
calib = pd.DataFrame(rows).T
display(calib.round(3))

,intercept,slope,"joint p (a=0, b=1)",mean(r2 / forecast)
"GARCH(1,1) (fixed)",0.256,0.616,0.286,1.001
EGARCH (fixed),-0.336,1.336,0.142,1.014
GJR-GARCH (fixed),-0.077,0.984,0.242,1.020
Rolling 21d var,0.483,0.423,0.000,1.256
"GARCH(1,1) (rolling)",0.244,0.638,0.349,1.001
EGARCH (rolling),-0.376,1.400,0.156,1.013
GJR-GARCH (rolling),-0.102,1.029,0.290,1.016
LSTM (old),-0.416,1.430,0.083,0.860
Hybrid (old),-0.701,1.948,0.138,0.940
LSTM (QLIKE),-0.620,1.955,0.457,0.992
